# RNS na ogbn-arxiv (E3)

Nasumican izbor cvorova sa indukovanim podgrafom (RNS), po uzoru na rad o implicitnoj
regularizaciji (arXiv 2605.22480), u nasem okviru i sa nasim protokolom, radi nezavisne
provere njihovog glavnog nalaza i merenja resursa koje taj rad ne daje.

Velicina batch-a je 20 odsto cvorova grafa, ista pokrivenost kao ClusterGCN K = 100 u
nasim ranijim merenjima, pa je poredjenje sa varijantama osnova i izolovano direktno.
Protokol identican Koraku 2: 5.000 koraka, ocena nad celim grafom na svakih 250 koraka,
GraphSAGE, L = 3, skriveni 256, lr 0,01, 10 seedova. Meri se i energija po treningu.

Procena trajanja: oko 45 minuta za svih deset seedova.


In [ ]:
SEEDOVI = [42, 123, 7, 1, 2, 3, 4, 5, 6, 8]
POKRIVENOST = 0.20
KORAKA, SLOJEVA, OCENA_SVAKIH = 5000, 3, 250


In [ ]:
import os, sys, time, json, gc, csv, random, shutil, logging, copy
from pathlib import Path

import torch
import torch.nn.functional as F
import numpy as np

from torch_geometric.nn import SAGEConv
from torch_geometric.loader import ClusterData, ClusterLoader, NeighborLoader
from torch_geometric.utils import to_undirected, subgraph
from ogb.nodeproppred import PygNodePropPredDataset
import torch_geometric.data

torch.serialization.add_safe_globals([
    torch_geometric.data.data.DataEdgeAttr,
    torch_geometric.data.data.DataTensorAttr,
    torch_geometric.data.storage.GlobalStorage,
])

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda')

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
                    handlers=[logging.StreamHandler(sys.stdout)], force=True)
log = logging.getLogger(__name__)

cc = torch.cuda.get_device_capability(0)
log.info(f'kartica {torch.cuda.get_device_name(0)}, compute capability {cc[0]}.{cc[1]}')

REZ = Path('rezultati_rns')
REZ.mkdir(parents=True, exist_ok=True)

try:
    import pynvml
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nvidia-ml-py'], check=False)
    import pynvml
try:
    pynvml.nvmlInit()
    _ruka = pynvml.nvmlDeviceGetHandleByIndex(0)
    log.info(f'NVML aktivan, trenutna snaga {pynvml.nvmlDeviceGetPowerUsage(_ruka)/1000:.0f} W')
except Exception:
    _ruka = None
    log.info('NVML nedostupan, energija se ne meri')


def energija_j():
    if _ruka is None:
        return float('nan')
    try:
        return pynvml.nvmlDeviceGetTotalEnergyConsumption(_ruka) / 1000.0
    except pynvml.NVMLError:
        return float('nan')


def snaga_w():
    if _ruka is None:
        return float('nan')
    try:
        return pynvml.nvmlDeviceGetPowerUsage(_ruka) / 1000.0
    except pynvml.NVMLError:
        return float('nan')


def postavi_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def direktorijum(seed):
    p = REZ / f'run_{seed}'
    p.mkdir(parents=True, exist_ok=True)
    return p


def upisi_red(putanja, red):
    novi = not putanja.exists()
    with open(putanja, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(red.keys()))
        if novi:
            w.writeheader()
        w.writerow(red)


def vec_uradjeno(putanja, kolona):
    if not putanja.exists():
        return set()
    with open(putanja) as f:
        return {r[kolona] for r in csv.DictReader(f)}


## Model i ucitavanje grafa

In [ ]:
SKRIVENI, LR = 256, 0.01


class GraphSAGE(torch.nn.Module):
    def __init__(self, ulaz, skriveni, izlaz, slojeva, dropout=0.5):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        self.slojeva, self.dropout = slojeva, dropout
        self.convs.append(SAGEConv(ulaz, skriveni))
        self.bns.append(torch.nn.BatchNorm1d(skriveni))
        for _ in range(slojeva - 2):
            self.convs.append(SAGEConv(skriveni, skriveni))
            self.bns.append(torch.nn.BatchNorm1d(skriveni))
        self.convs.append(SAGEConv(skriveni, izlaz))

    def forward(self, x, edge_index):
        for i in range(self.slojeva - 1):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.convs[-1](x, edge_index)


def trening_korak(model, batch, opt):
    batch = batch.to(device, non_blocking=True)
    opt.zero_grad()
    out = model(batch.x, batch.edge_index)
    maska = batch.train_mask
    if maska.sum() == 0:
        return
    gubitak = F.cross_entropy(out[maska], batch.y[maska])
    gubitak.backward()
    opt.step()


In [ ]:
from ogb.nodeproppred import Evaluator

dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='podaci')
data = dataset[0]
data.y = data.y.squeeze()
split_idx = dataset.get_idx_split()
evaluator = Evaluator(name='ogbn-arxiv')
data.edge_index = to_undirected(data.edge_index)
data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[split_idx['train']] = True
B = round(data.num_nodes * POKRIVENOST)
log.info(f'ogbn-arxiv: {data.num_nodes:,} cvorova, batch {B:,} cvorova ({POKRIVENOST*100:.0f}%)')


@torch.no_grad()
def oceni(model):
    model.eval()
    x = data.x.to(device, non_blocking=True)
    ei = data.edge_index.to(device, non_blocking=True)
    out = model(x, ei).argmax(dim=-1, keepdim=True).cpu()
    del x, ei
    torch.cuda.empty_cache()
    return {s: evaluator.eval({'y_true': data.y[split_idx[s]].unsqueeze(1),
                               'y_pred': out[split_idx[s]]})['acc']
            for s in ['train', 'valid', 'test']}


## Merenje

In [ ]:
for SEED in SEEDOVI:
    fajl = direktorijum(SEED) / 'rns.csv'
    if f'RNS_B{B}' in vec_uradjeno(fajl, 'oznaka'):
        continue
    postavi_seed(SEED)
    model = GraphSAGE(data.num_features, SKRIVENI, dataset.num_classes, SLOJEVA).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    naj_val, naj_test, naj_korak = 0.0, float('nan'), 0
    vrh, trening_vreme, korak = 0.0, 0.0, 0
    potega_l, snage = [], []
    e0 = energija_j()
    torch.cuda.reset_peak_memory_stats()
    while korak < KORAKA:
        idx = torch.randperm(data.num_nodes)[:B]
        ei_b, _ = subgraph(idx, data.edge_index, relabel_nodes=True,
                           num_nodes=data.num_nodes)
        torch.cuda.synchronize()
        t0 = time.time()
        x = data.x[idx].to(device, non_blocking=True)
        y = data.y[idx].to(device, non_blocking=True)
        maska = data.train_mask[idx].to(device, non_blocking=True)
        ei = ei_b.to(device, non_blocking=True)
        opt.zero_grad()
        out = model(x, ei)
        gubitak = F.cross_entropy(out[maska], y[maska])
        gubitak.backward()
        opt.step()
        torch.cuda.synchronize()
        vrh = max(vrh, torch.cuda.max_memory_allocated() / 1e6)
        trening_vreme += time.time() - t0
        snage.append(snaga_w())
        if len(potega_l) < 20:
            potega_l.append(ei_b.size(1))
        korak += 1
        if korak % OCENA_SVAKIH == 0:
            a = oceni(model)
            if a['valid'] > naj_val:
                naj_val, naj_test, naj_korak = a['valid'], a['test'], korak
            model.train()
            torch.cuda.reset_peak_memory_stats()
    e1 = energija_j()

    upisi_red(fajl, {
        'oznaka': f'RNS_B{B}', 'cvorova_po_koraku': B,
        'potega_po_koraku': float(np.mean(potega_l)),
        'ukupno_koraka': korak, 'naj_val': naj_val, 'naj_test': naj_test,
        'naj_korak': naj_korak,
        'vreme_po_koraku_ms': trening_vreme / korak * 1000,
        'memorija_mb': vrh, 'energija_j': e1 - e0,
        'snaga_prosek_w': float(np.nanmean(snage)),
        'snaga_min_w': float(np.nanmin(snage)),
        'snaga_max_w': float(np.nanmax(snage))})
    log.info(f'seed {SEED} | RNS: test={naj_test*100:.2f}% (korak {naj_korak}), '
             f'{trening_vreme/korak*1000:.1f} ms/korak, vrh {vrh:.0f} MB, {e1-e0:.0f} J')
    del model, opt
    gc.collect()
    torch.cuda.empty_cache()

t = []
for SEED in SEEDOVI:
    p = direktorijum(SEED) / 'rns.csv'
    if p.exists():
        with open(p) as f:
            for r in csv.DictReader(f):
                t.append(float(r['naj_test']) * 100)
if len(t) > 1:
    log.info(f'RNS: test {np.mean(t):.2f} +/- {np.std(t, ddof=1):.2f} % preko {len(t)} seedova')


## Arhiviranje

In [ ]:
import zipfile

arhiva = Path('rezultati_rns.zip')
with zipfile.ZipFile(arhiva, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(REZ.rglob('*')):
        if p.is_file():
            z.write(p)
log.info(f'{arhiva}: {arhiva.stat().st_size / 1e6:.1f} MB, preuzeti i sacuvati u projekat')
